<a href="https://colab.research.google.com/github/mloulaps-rgb/---/blob/main/HCI_%D0%9B%D0%A06_%D0%A7%D0%B0%D1%81%D1%82%D0%B8%D0%BD%D0%BE%D0%BC%D0%BE%D0%B2%D0%BD%D0%B8%D0%B9_%D0%B0%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_HMM_%D0%92%D0%B5%D0%BB%D0%B8%D1%87%D0%BA%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Крок 0. Службові методи, морфологічні правила та завантаження даних.
import os, urllib.request

# Завантаження навчального й тестового корпусів (Wall Street Journal, DeepLearning.AI)
os.makedirs("data", exist_ok=True)
BASE = ("https://raw.githubusercontent.com/amanchadha/"
        "coursera-natural-language-processing-specialization/master/"
        "2%20-%20Natural%20Language%20Processing%20with%20Probabilistic%20Models/"
        "Week%202/")
for fname in ["WSJ_02-21.pos", "WSJ_24.pos", "hmm_vocab.txt", "test.words"]:
    dst = os.path.join("data", fname)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(BASE + fname.replace(" ", "%20"), dst)
    print(f"{fname}: {os.path.getsize(dst)} байт")

import string
# Символи пунктуації
punct = set(string.punctuation)

# Морфологічні правила для класифікації невідомих слів
noun_suffix = ["action", "age", "ance", "cy", "dom", "ee", "ence", "er", "hood", "ion",
               "ism", "ist", "ity", "ling", "ment", "ness", "or", "ry", "scape", "ship", "ty"]
verb_suffix = ["ate", "ify", "ise", "ize"]
adj_suffix  = ["able", "ese", "ful", "i", "ian", "ible", "ic", "ish", "ive", "less", "ly", "ous"]
adv_suffix  = ["ward", "wards", "wise"]


def assign_unk(tok):
    """Призначення мітки для невідомого слова за морфологічними ознаками."""
    if any(char.isdigit() for char in tok):
        return "--unk_digit--"
    elif any(char in punct for char in tok):
        return "--unk_punct--"
    elif any(char.isupper() for char in tok):
        return "--unk_upper--"
    elif any(tok.endswith(suffix) for suffix in noun_suffix):
        return "--unk_noun--"
    elif any(tok.endswith(suffix) for suffix in verb_suffix):
        return "--unk_verb--"
    elif any(tok.endswith(suffix) for suffix in adj_suffix):
        return "--unk_adj--"
    elif any(tok.endswith(suffix) for suffix in adv_suffix):
        return "--unk_adv--"
    return "--unk--"


def preprocess(vocab, data_fp):
    """Попередня обробка тестових даних: обробка кінців речень і невідомих слів."""
    orig, prep = [], []
    with open(data_fp, "r") as data_file:
        for word in data_file:
            if not word.split():                       # кінець речення
                orig.append(word.strip()); prep.append("--n--"); continue
            elif word.strip() not in vocab:            # невідоме слово
                orig.append(word.strip()); prep.append(assign_unk(word.strip())); continue
            else:
                orig.append(word.strip()); prep.append(word.strip())
    return orig, prep

print("Службові функції визначено.")


WSJ_02-21.pos: 8279089 байт
WSJ_24.pos: 286063 байт
hmm_vocab.txt: 196571 байт
test.words: 180266 байт
Службові функції визначено.


In [2]:
# Крок 1. Підготовка навчальних та тестових даних.
import numpy as np
import pandas as pd
from collections import defaultdict
import math

# Навчальний корпус (слово\tтег)
with open("./data/WSJ_02-21.pos", "r") as f:
    training_corpus = f.readlines()

# Словник відомих слів
with open("./data/hmm_vocab.txt", "r") as f:
    voc_l = f.read().split("\n")

vocab = {}
for i, word in enumerate(sorted(voc_l)):
    vocab[word] = i

# Тестовий корпус з істинними тегами
with open("./data/WSJ_24.pos", "r") as f:
    y = f.readlines()

# Попередня обробка тестових слів
_, prep = preprocess(vocab, "./data/test.words")

print("Рядків у навчальному корпусі :", len(training_corpus))
print("Розмір словника vocab        :", len(vocab))
print("Рядків у тестовому корпусі y  :", len(y))
print("Оброблених тестових слів prep :", len(prep))
print("\nПерші рядки навчального корпусу:")
for line in training_corpus[:5]:
    print("  ", repr(line))


Рядків у навчальному корпусі : 989860
Розмір словника vocab        : 23777
Рядків у тестовому корпусі y  : 34199
Оброблених тестових слів prep : 34199

Перші рядки навчального корпусу:
   'In\tIN\n'
   'an\tDT\n'
   'Oct.\tNNP\n'
   '19\tCD\n'
   'review\tNN\n'


In [3]:
# Крок 2. Створення словників частот переходів, емісій та тегів.
def get_word_tag(line, vocab):
    """Отримання (слово, тег) з рядка корпусу."""
    if not line.split():
        return "--n--", "--s--"
    parts = line.split()
    if len(parts) >= 2:
        word, tag = parts[0], parts[1]
        if word not in vocab:
            word = assign_unk(word)
        return word, tag
    return "--n--", "--s--"


def create_dictionaries(training_corpus, vocab):
    """Створення словників emission_counts, transition_counts, tag_counts."""
    emission_counts   = defaultdict(int)
    transition_counts = defaultdict(int)
    tag_counts        = defaultdict(int)
    prev_tag = "--s--"
    for word_tag in training_corpus:
        word, tag = get_word_tag(word_tag, vocab)
        transition_counts[(prev_tag, tag)] += 1
        emission_counts[(tag, word)] += 1
        tag_counts[tag] += 1
        prev_tag = tag
    return emission_counts, transition_counts, tag_counts


emission_counts, transition_counts, tag_counts = create_dictionaries(training_corpus, vocab)
states = sorted(tag_counts.keys())

print("Кількість POS-тегів (станів):", len(tag_counts))
print("Пар переходів transition_counts:", len(transition_counts))
print("Пар емісій emission_counts     :", len(emission_counts))
print("\nПриклади тегів та їх частот:")
for t in states[:12]:
    print(f"  {t:8s} -> {tag_counts[t]}")
print("\nПриклади переходів:")
for k in [('--s--','NN'), ('NN','VB'), ('DT','NN'), ('JJ','NN')]:
    print(f"  {k} -> {transition_counts[k]}")
print("\nПриклади емісій:")
for k in [('NN','year'), ('DT','the'), ('VB','be')]:
    print(f"  {k} -> {emission_counts[k]}")


Кількість POS-тегів (станів): 46
Пар переходів transition_counts: 1421
Пар емісій emission_counts     : 31140

Приклади тегів та їх частот:
  #        -> 142
  $        -> 7372
  ''       -> 6919
  (        -> 1366
  )        -> 1376
  ,        -> 48727
  --s--    -> 39832
  .        -> 39478
  :        -> 4772
  CC       -> 23947
  CD       -> 36568
  DT       -> 81842

Приклади переходів:
  ('--s--', 'NN') -> 1598
  ('NN', 'VB') -> 186
  ('DT', 'NN') -> 38873
  ('JJ', 'NN') -> 27550

Приклади емісій:
  ('NN', 'year') -> 2239
  ('DT', 'the') -> 41098
  ('VB', 'be') -> 3724


In [4]:
# Крок 3-4. Обчислення матриці переходів A та матриці емісій B.
def create_transition_matrix(alpha, tag_counts, transition_counts):
    """Матриця переходів A зі згладжуванням Лапласа."""
    all_tags = sorted(tag_counts.keys())
    num_tags = len(all_tags)
    A = np.zeros((num_tags, num_tags))
    for i in range(num_tags):
        count_prev_tag = tag_counts[all_tags[i]]
        for j in range(num_tags):
            key = (all_tags[i], all_tags[j])
            count = transition_counts[key] if key in transition_counts else 0
            A[i, j] = (count + alpha) / (count_prev_tag + alpha * num_tags)
    return A


def create_emission_matrix(alpha, tag_counts, emission_counts, vocab):
    """Матриця емісій B зі згладжуванням Лапласа."""
    all_tags = sorted(tag_counts.keys())
    num_tags = len(tag_counts)
    num_words = len(vocab)
    B = np.zeros((num_tags, num_words))
    for i in range(num_tags):
        count_tag = tag_counts[all_tags[i]]
        for j in range(num_words):
            key = (all_tags[i], vocab[j])
            count = emission_counts[key] if key in emission_counts else 0
            B[i, j] = (count + alpha) / (count_tag + alpha * num_words)
    return B


alpha = 0.001
# Індексований словник слів (слово"->"індекс), узгоджений із порядком стовпців B
words_sorted = sorted(vocab.keys())
word_to_index = {word: i for i, word in enumerate(words_sorted)}

A = create_transition_matrix(alpha, tag_counts, transition_counts)
B = create_emission_matrix(alpha, tag_counts, emission_counts, words_sorted)

print("Розмір матриці переходів A:", A.shape)
print("Розмір матриці емісій   B:", B.shape)

# Фрагмент матриці переходів для наочності
sub_tags = ['--s--', 'DT', 'JJ', 'NN', 'VB']
idx = [states.index(t) for t in sub_tags]
df_A = pd.DataFrame(A[np.ix_(idx, idx)], index=sub_tags, columns=sub_tags)
print("\nФрагмент матриці переходів A:")
print(df_A.round(4))


Розмір матриці переходів A: (46, 46)
Розмір матриці емісій   B: (46, 23777)

Фрагмент матриці переходів A:
        --s--      DT      JJ      NN      VB
--s--  0.0000  0.2171  0.0420  0.0401  0.0029
DT     0.0001  0.0016  0.2181  0.4750  0.0003
JJ     0.0001  0.0034  0.0747  0.4500  0.0001
NN     0.0005  0.0068  0.0088  0.1222  0.0014
VB     0.0002  0.2213  0.0847  0.0616  0.0049


In [5]:
# Крок 5. Алгоритм Вітербі: ініціалізація, пряме та зворотне проходження.
def initialize(states, tag_counts, A, B, corpus, vocab):
    """Ініціалізація матриць best_probs та best_paths (перший стовпець)."""
    num_tags = len(tag_counts)
    best_probs = np.zeros((num_tags, len(corpus)))
    best_paths = np.zeros((num_tags, len(corpus)), dtype=int)
    s_idx = states.index("--s--")
    word_idx = vocab.get(corpus[0], vocab.get("--unk--", 0))
    for i in range(num_tags):
        if A[s_idx, i] == 0:
            best_probs[i, 0] = float("-inf")
        else:
            best_probs[i, 0] = math.log(A[s_idx, i]) + math.log(B[i, word_idx])
    return best_probs, best_paths


def viterbi_forward(A, B, corpus, best_probs, best_paths, vocab):
    """Пряме проходження: обчислення найкращих ймовірностей та шляхів."""
    num_tags = best_probs.shape[0]
    for i in range(1, len(corpus)):
        word_idx = vocab.get(corpus[i], vocab.get("--unk--", 0))
        logB = np.log(B[:, word_idx])
        for j in range(num_tags):
            # векторизований внутрішній цикл по k (ідентичний за результатом)
            probs = best_probs[:, i-1] + np.log(A[:, j]) + logB[j]
            k = int(np.argmax(probs))
            best_probs[j, i] = probs[k]
            best_paths[j, i] = k
    return best_probs, best_paths


def viterbi_backward(best_probs, best_paths, corpus, states):
    """Зворотне проходження: відновлення найкращої послідовності тегів."""
    m = best_paths.shape[1]
    z = [None] * m
    pred = [None] * m
    best_last = float("-inf")
    for k in range(best_probs.shape[0]):
        if best_probs[k, m-1] > best_last:
            best_last = best_probs[k, m-1]
            z[m-1] = k
    pred[m-1] = states[z[m-1]]
    for i in range(m-2, -1, -1):
        z[i] = best_paths[z[i+1], i+1]
        pred[i] = states[z[i]]
    return pred

print("Функції алгоритму Вітербі визначено.")


Функції алгоритму Вітербі визначено.


In [6]:
# Крок 6. Запуск HMM-моделі та обчислення точності.
def compute_accuracy(pred, y):
    """Точність POS-тегування: частка правильно передбачених тегів."""
    num_correct = 0
    total = 0
    for prediction, y_item in zip(pred, y):
        word_tag_tuple = y_item.strip().split("\t")
        if len(word_tag_tuple) != 2:
            continue
        word, tag = word_tag_tuple
        if tag == prediction:
            num_correct += 1
        total += 1
    return num_correct / total


import time
t0 = time.time()
best_probs, best_paths = initialize(states, tag_counts, A, B, prep, word_to_index)
best_probs, best_paths = viterbi_forward(A, B, prep, best_probs, best_paths, word_to_index)
pred = viterbi_backward(best_probs, best_paths, prep, states)
elapsed = time.time() - t0

accuracy = compute_accuracy(pred, y)
print(f"Точність власної HMM-моделі (Вітербі): {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Час тегування {len(prep)} слів: {elapsed:.1f} с")

print("\nПриклад передбачених тегів:")
for i in range(8):
    print(f"  {prep[i]:14s} -> {pred[i]}")


Точність власної HMM-моделі (Вітербі): 0.9545 (95.45%)
Час тегування 34199 слів: 15.0 с

Приклад передбачених тегів:
  The            -> DT
  economy        -> NN
  's             -> POS
  temperature    -> NN
  will           -> MD
  be             -> VB
  taken          -> VBN
  from           -> IN


In [7]:
# Крок 7. Порівняльний аналіз із POS-тегуванням на основі NLTK.
import nltk
for res in ["punkt", "punkt_tab", "averaged_perceptron_tagger",
            "averaged_perceptron_tagger_eng", "treebank"]:
    try: nltk.download(res, quiet=True)
    except Exception: pass

from nltk import word_tokenize
from nltk.corpus import treebank
from nltk.tag import UnigramTagger, BigramTagger

text = "The economy's temperature will be taken from several vantage points this week."
tokens = word_tokenize(text)
nltk_tags = nltk.pos_tag(tokens)
print("POS-тегування NLTK (вбудований перцептрон):")
for word, tag in nltk_tags:
    print(f"  {word:14s} -> {tag}")

# Навчання та оцінка n-грамних тегерів NLTK на корпусі treebank
train_data = treebank.tagged_sents()[:3000]
test_data  = treebank.tagged_sents()[3000:]
unigram_tagger = UnigramTagger(train_data)
bigram_tagger  = BigramTagger(train_data, backoff=unigram_tagger)

try:
    nltk_accuracy = bigram_tagger.accuracy(test_data)
except AttributeError:
    nltk_accuracy = bigram_tagger.evaluate(test_data)

print(f"\nТочність NLTK BigramTagger (backoff Unigram): {nltk_accuracy:.4f} ({nltk_accuracy*100:.2f}%)")
print(f"Точність власної HMM-реалізації (Вітербі)  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Різниця точності                           : {abs(accuracy - nltk_accuracy):.4f}")
print("\nВисновок: власна HMM з алгоритмом Вітербі враховує контекст переходів між")
print("тегами й демонструє вищу точність, ніж n-грамний BigramTagger NLTK,")
print("навчений на меншому корпусі treebank.")


POS-тегування NLTK (вбудований перцептрон):
  The            -> DT
  economy        -> NN
  's             -> POS
  temperature    -> NN
  will           -> MD
  be             -> VB
  taken          -> VBN
  from           -> IN
  several        -> JJ
  vantage        -> NN
  points         -> NNS
  this           -> DT
  week           -> NN
  .              -> .

Точність NLTK BigramTagger (backoff Unigram): 0.8648 (86.48%)
Точність власної HMM-реалізації (Вітербі)  : 0.9545 (95.45%)
Різниця точності                           : 0.0897

Висновок: власна HMM з алгоритмом Вітербі враховує контекст переходів між
тегами й демонструє вищу точність, ніж n-грамний BigramTagger NLTK,
навчений на меншому корпусі treebank.
